In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, accuracy_score
from tabulate import tabulate
import time
import tracemalloc
import matplotlib.pyplot as plt



In [4]:
# Step 1:Load dataset
df = pd.read_csv("/content/sample_data/fraud_detection.csv")

# Check for missing values in the dataset
missing_values = df.isnull().sum()
print("Missing Values:\n", missing_values)

# Drop rows with missing values (if any)
df = df.dropna()

# Prepare the data
X = df.drop(columns=["Class", "Time"])  # Features
y = df["Class"]  # Target

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=50, shuffle=True
)


Missing Values:
 Time      0
V1        0
V2        0
V3        0
V4        0
V5        0
V6        0
V7        0
V8        0
V9        0
V10       0
V11       0
V12       0
V13       1
V14       1
V15       1
V16       1
V17       1
V18       1
V19       1
V20       1
V21       1
V22       1
V23       1
V24       1
V25       1
V26       1
V27       1
V28       1
Amount    1
Class     1
dtype: int64


In [5]:
# Step 2: KNN Model with Cross-Validation
print("\nKNN Model Cross-Validation")
knn_neighbors = [5, 10, 15]
knn_results = {}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=50)

for n in knn_neighbors:
    knn = KNeighborsClassifier(n_neighbors=n)

    # Cross-validation accuracy scores
    cv_scores = cross_val_score(knn, X_train, y_train, cv=cv, scoring="accuracy")

    # Final training and testing performance
    knn.fit(X_train, y_train)
    train_acc = knn.score(X_train, y_train)
    test_acc = knn.score(X_test, y_test)

    knn_results[n] = {
        "CV Accuracy": cv_scores.mean(),
        "Training Accuracy": train_acc,
        "Testing Accuracy": test_acc,
        "Bias": 1 - train_acc,
        "Variance": train_acc - test_acc,
    }
    print(f"KNN (n_neighbors={n}): CV Accuracy={cv_scores.mean():.4f}, "
          f"Training Accuracy={train_acc:.4f}, Testing Accuracy={test_acc:.4f}")


KNN Model Cross-Validation
KNN (n_neighbors=5): CV Accuracy=0.9991, Training Accuracy=0.9993, Testing Accuracy=0.9993
KNN (n_neighbors=10): CV Accuracy=0.9990, Training Accuracy=0.9991, Testing Accuracy=0.9990
KNN (n_neighbors=15): CV Accuracy=0.9989, Training Accuracy=0.9991, Testing Accuracy=0.9990


In [6]:
# Step 3.1: Silhouette Analysis for n_clusters = [2, 3, 4]
print("\nSilhouette Analysis for n_clusters = [2, 3, 4]")
silhouette_scores = []
for n_clusters in [2, 3, 4]:
    kmeans = KMeans(n_clusters=n_clusters, random_state=50)
    cluster_labels = kmeans.fit_predict(X_train)
    silhouette_avg = silhouette_score(X_train, cluster_labels)
    silhouette_scores.append((n_clusters, silhouette_avg))
    print(f"n_clusters={n_clusters}: Average Silhouette Score = {silhouette_avg:.4f}")


Silhouette Analysis for n_clusters = [2, 3, 4]
n_clusters=2: Average Silhouette Score = 0.9146
n_clusters=3: Average Silhouette Score = 0.8609
n_clusters=4: Average Silhouette Score = 0.8064


In [7]:
# Step 3.2: Cross-Validation for K-Means with "lloyd" and "elkan"
print("\nK-Means Cross-Validation for Algorithms 'lloyd' and 'elkan' with n_clusters=2")
kmeans_algorithms = ["lloyd", "elkan"]
kmeans_results = {}

for algo in kmeans_algorithms:
    silhouette_train_scores = []
    silhouette_test_scores = []
    train_times = []
    test_times = []
    memory_usages = []

    for train_idx, test_idx in cv.split(X_train, y_train):
        X_train_cv, X_test_cv = X_train.iloc[train_idx], X_train.iloc[test_idx]

        # Training time and memory tracking
        tracemalloc.start()
        start = time.time()
        kmeans = KMeans(n_clusters=2, algorithm=algo, random_state=50)
        kmeans.fit(X_train_cv)
        train_time = time.time() - start
        train_memory = tracemalloc.get_traced_memory()[1] / 1024  # Peak memory usage
        tracemalloc.stop()

        # Silhouette scores
        train_labels = kmeans.predict(X_train_cv)
        test_labels = kmeans.predict(X_test_cv)
        silhouette_train_scores.append(silhouette_score(X_train_cv, train_labels))
        silhouette_test_scores.append(silhouette_score(X_test_cv, test_labels))

        # Testing time and memory tracking
        tracemalloc.start()
        start = time.time()
        _ = kmeans.predict(X_test_cv)
        test_time = time.time() - start
        test_memory = tracemalloc.get_traced_memory()[1] / 1024
        tracemalloc.stop()

        train_times.append(train_time)
        test_times.append(test_time)
        memory_usages.append((train_memory, test_memory))

    kmeans_results[algo] = {
        "Train Silhouette (Mean)": np.mean(silhouette_train_scores),
        "Test Silhouette (Mean)": np.mean(silhouette_test_scores),
        "Training Time (Mean)": np.mean(train_times),
        "Testing Time (Mean)": np.mean(test_times),
        "Training Memory (Mean KB)": np.mean([m[0] for m in memory_usages]),
        "Testing Memory (Mean KB)": np.mean([m[1] for m in memory_usages]),
    }

    print(f"Algorithm={algo}: Train Silhouette={np.mean(silhouette_train_scores):.4f}, "
          f"Test Silhouette={np.mean(silhouette_test_scores):.4f}, "
          f"Training Time={np.mean(train_times):.4f}s, "
          f"Testing Time={np.mean(test_times):.4f}s, "
          f"Training Memory={np.mean([m[0] for m in memory_usages]):.2f} KB, "
          f"Testing Memory={np.mean([m[1] for m in memory_usages]):.2f} KB")


K-Means Cross-Validation for Algorithms 'lloyd' and 'elkan' with n_clusters=2
Algorithm=lloyd: Train Silhouette=0.9146, Test Silhouette=0.9146, Training Time=0.8028s, Testing Time=0.0092s, Training Memory=51251.62 KB, Testing Memory=6734.26 KB
Algorithm=elkan: Train Silhouette=0.9146, Test Silhouette=0.9146, Training Time=0.5313s, Testing Time=0.0070s, Training Memory=51251.38 KB, Testing Memory=6734.14 KB


In [8]:
# Step 4: Model Comparison
print("\nComparison of Models")
comparison_data = []

# KNN Results
best_knn = max(knn_results, key=lambda n: knn_results[n]["Testing Accuracy"])
comparison_data.append([
    "KNN",
    f"n_neighbors={best_knn}",
    knn_results[best_knn]["CV Accuracy"],
    knn_results[best_knn]["Training Accuracy"],
    knn_results[best_knn]["Testing Accuracy"],
    knn_results[best_knn]["Bias"],
    knn_results[best_knn]["Variance"],
    "N/A", "N/A", "N/A"
])

# K-Means Results
best_kmeans = max(kmeans_results, key=lambda algo: kmeans_results[algo]["Train Silhouette (Mean)"])
comparison_data.append([
    "K-Means",
    f"Algorithm={best_kmeans}",
    kmeans_results[best_kmeans]["Train Silhouette (Mean)"],
    kmeans_results[best_kmeans]["Test Silhouette (Mean)"],
    "N/A", "N/A",
    kmeans_results[best_kmeans]["Training Time (Mean)"],
    kmeans_results[best_kmeans]["Testing Time (Mean)"],
    kmeans_results[best_kmeans]["Training Memory (Mean KB)"],
    kmeans_results[best_kmeans]["Testing Memory (Mean KB)"]
])

# Print Comparison Table
headers = ["Model", "Hyperparameters", "CV Accuracy/Score", "Training Accuracy/Score",
           "Testing Accuracy/Score", "Bias", "Variance", "Training Time (s)",
           "Testing Time (s)", "Training Memory (KB)", "Testing Memory (KB)"]
print(tabulate(comparison_data, headers=headers, tablefmt="grid"))



Comparison of Models
+---------+-------------------+---------------------+---------------------------+--------------------------+-----------------------+--------------+----------------------+--------------------+------------------------+
| Model   | Hyperparameters   |   CV Accuracy/Score |   Training Accuracy/Score | Testing Accuracy/Score   | Bias                  |     Variance | Training Time (s)    | Testing Time (s)   | Training Memory (KB)   |
+=========+===================+=====================+===========================+==========================+=======================+==============+======================+====================+========================+
| KNN     | n_neighbors=5     |            0.999108 |                  0.999256 | 0.9992988420269839       | 0.0007436523956231289 | -4.24944e-05 | N/A                  | N/A                | N/A                    |
+---------+-------------------+---------------------+---------------------------+--------------------------+--